<a href="https://colab.research.google.com/github/Danah911/2026-GP1-16/blob/main/REPLICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REPLICA: A Continuous-Time Digital Twin for Forecasting CDR-SB in ADNI

Alzheimer's disease progresses at a different pace in every patient, and clinic visits arrive at irregular intervals with different tests performed each time. In this project we build **REPLICA**, a digital twin that reads a patient's history of clinical scores and forecasts their Clinical Dementia Rating Sum of Boxes (**CDR-SB**, 0–18) 6, 12 and 24 months ahead, together with a calibrated 90% prediction interval that tightens as new visits arrive.

We use the Alzheimer's Disease Neuroimaging Initiative (**ADNI**) dataset. Our pipeline cleans 22 raw ADNI tables without imputing a single value, builds leakage-free patient sequences, trains a **GRU-ODE-Bayes** model that evolves a hidden state in continuous time between visits and corrects it at each visit, calibrates its uncertainty on held-out patients, and compares its forecasts against two clinically meaningful baselines on patients the model has never seen.

**Workflow**

1. Setup and configuration
2. Data loading
3. Data cleaning
4. Building the patient visit table
5. Exploratory data analysis
6. Preprocessing: patient-level split, normalization and evaluation examples
7. The GRU-ODE-Bayes model
8. Training
9. Evaluation: baselines, calibration and test results
10. The digital twin in use
11. Conclusion

**How to run.** Open the notebook in Google Colab with a GPU runtime, upload `Dataset for Model.rar` (or the extracted folder), set `INPUT_PATH` in the configuration cell, and select *Runtime → Run all*. Every output is written to `OUTPUT_DIR` and zipped at the end.

## 1. Setup and Configuration

We start by loading the libraries, fixing the random seed and collecting every choice of the pipeline in one configuration cell, so that thresholds, paths and model sizes can be changed in a single place without touching the code below. We also define a small set of display helpers that give every table and figure the same visual style.

In [ ]:
import sys, subprocess, importlib.util, os, re, json, math, copy, html, hashlib, shutil, zipfile, platform, warnings
from pathlib import Path
from types import SimpleNamespace

for package in ['pyreadr']:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

import numpy as np
import pandas as pd
import pyreadr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import norm as gaussian
import matplotlib.pyplot as plt
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Python {platform.python_version()} · pandas {pd.__version__} · torch {torch.__version__} · device: {DEVICE}')

In [ ]:
CFG = SimpleNamespace(
    # ---- paths ----
    INPUT_PATH  = '/content/Dataset for Model.rar',      # .rar, .zip, or an extracted folder of .rda files
    OUTPUT_DIR  = '/content/REPLICA_Output',
    AS_OF_DATE  = '2026-10-14',                          # date of the ADNI data release
    SEED        = 42,
    # ---- sequence construction ----
    CHANNELS    = ['cdrsb', 'mmse', 'adas13', 'faq'],    # time-varying inputs; cdrsb is also the forecast target
    STATIC      = ['baseline_age', 'female', 'education', 'apoe4_count'],
    MIN_VISITS  = 2,                                     # patients need at least this many visits
    MAX_GAP_DAYS = 90,                                   # tolerance when linking a diagnosis to a visit
    JUMP_UP = 4.0, JUMP_DAYS = 183, JUMP_DOWN = 3.0,     # implausible CDR-SB jumps are flagged, never deleted
    # ---- split and evaluation examples ----
    SPLIT_RATIOS = {'train': 0.70, 'val': 0.10, 'calib': 0.10, 'test': 0.10},
    VISIT_BINS  = [2, 3, 5, 8],                          # strata: 2 | 3-4 | 5-7 | 8+ visits
    HORIZONS    = {6: 2, 12: 3, 24: 4},                  # forecast horizon in months: +/- window in months
    EVAL_DX     = 'MCI',                                 # diagnosis at the index visit (selection only, never an input)
    MIN_CDR_OBS_EVAL = 2,                                # CDR-SB observations required up to the index visit
    # ---- model ----
    HIDDEN_SIZE = 32,
    ODE_STEP_YEARS = 1 / 12, MAX_SUBSTEPS = 12,          # monthly ODE steps, at most 12 per gap
    LOGVAR_MIN = -6.0, LOGVAR_MAX = 4.0,
    # ---- training ----
    BATCH_SIZE = 64, LEARNING_RATE = 1e-3, WEIGHT_DECAY = 1e-5, GRAD_CLIP = 5.0,
    MAX_EPOCHS = 100, PATIENCE = 10,
    # ---- uncertainty ----
    COVERAGE = 0.90,
    RELIABILITY_LEVELS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95],
)
CHANNELS, STATICS, SPLITS = list(CFG.CHANNELS), list(CFG.STATIC), list(CFG.SPLIT_RATIOS)
HORIZONS = sorted(CFG.HORIZONS)
AS_OF_DATE = pd.Timestamp(CFG.AS_OF_DATE)
INPUT_PATH, OUT = Path(CFG.INPUT_PATH), Path(CFG.OUTPUT_DIR)
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)

In [ ]:
PURPLE, MID, LIGHT, BAND, GREY, INK = '#594575', '#8a6fb0', '#b89fd4', '#e9dff4', '#8c8794', '#423250'
PALETTE = [PURPLE, MID, LIGHT, '#d7c5e9', INK, '#a98cc9']
CSS = ('<style>'
       '.replica {font-family:Arial,sans-serif; color:#302b3a; max-width:1100px; line-height:1.55;}'
       '.replica h3 {font-size:17px; color:#594575; margin:18px 0 10px;}'
       '.replica table {border-collapse:collapse; width:100%; font-size:13px; margin:8px 0 16px;}'
       '.replica th {background:#e9dff4; color:#423250; padding:9px 12px; text-align:left; border-bottom:1px solid #d7c5e9;}'
       '.replica td {padding:7px 12px; border-bottom:1px solid #eee8f4; text-align:left; overflow-wrap:anywhere;}'
       '.replica tr:nth-child(even) {background:#faf7fd;}'
       '.replica p {color:#655c70; font-size:13px;}'
       '</style>')

def show(frame, title, limit=8):
    """Display a DataFrame as a styled table (no index, rounded numbers, limited rows)."""
    view = frame.head(limit).copy()
    for column in view.columns:
        if pd.api.types.is_datetime64_any_dtype(view[column]):
            view[column] = view[column].dt.strftime('%Y-%m-%d')
        elif isinstance(view[column].dtype, pd.api.extensions.ExtensionDtype):
            view[column] = view[column].astype(object).where(view[column].notna(), np.nan)
    body = '<div class="replica"><h3>' + html.escape(str(title)) + '</h3>'
    body += view.to_html(index=False, border=0, na_rep='—',
                         float_format=lambda v: f'{v:,.3f}'.rstrip('0').rstrip('.') if abs(v) < 1 else f'{v:,.2f}')
    if len(frame) > limit:
        body += f'<p>Showing {limit} of {len(frame):,} rows.</p>'
    display(HTML(CSS + body + '</div>'))

def note(message):
    display(HTML(CSS + '<div class="replica"><p>' + html.escape(message) + '</p></div>'))

def style_axis(ax, title=None):
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    ax.grid(axis='y', color='#eee8f4', linewidth=0.8)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, color=PURPLE, fontsize=11, loc='left')

def finish(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / 'figures' / f'{name}.png', dpi=130, bbox_inches='tight')
    plt.show()

## 2. Data Loading

ADNI ships its tables as R data files (`.rda`). We extract the archive, read all 22 tables into memory and record the row count and SHA-256 hash of every source file, so the exact data release behind every result can be traced. Sixteen tables hold patient measurements; the remaining six (`REGISTRY`, `VISITS`, `DATADIC`, …) are reference tables that we use to interpret fields, not as measurements.

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Input not found: {INPUT_PATH}. Set CFG.INPUT_PATH to the upload location.')

if INPUT_PATH.is_file() and INPUT_PATH.suffix.lower() in ['.rar', '.zip']:
    DATA_DIR = OUT / 'extracted'
    DATA_DIR.mkdir(exist_ok=True)
    if INPUT_PATH.suffix.lower() == '.rar':
        if sys.platform.startswith('linux') and not shutil.which('bsdtar'):
            subprocess.run(['apt-get', '-qq', 'update'], check=True, stdout=subprocess.DEVNULL)
            subprocess.run(['apt-get', '-qq', 'install', '-y', 'libarchive-tools'], check=True, stdout=subprocess.DEVNULL)
        reader = shutil.which('bsdtar')
        if reader is None:
            raise RuntimeError('RAR input needs bsdtar; alternatively extract the folder before running.')
        entries = subprocess.run([reader, '-tf', str(INPUT_PATH)], check=True, capture_output=True, text=True).stdout.splitlines()
        for entry in entries:
            target = (DATA_DIR / entry.replace('\\', '/')).resolve()
            if not target.is_relative_to(DATA_DIR.resolve()):
                raise ValueError('Unsafe archive path: ' + entry)
            if target.suffix.lower() in ['.rda', '.rdata', '.csv']:
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open('wb') as destination:
                    subprocess.run([reader, '-xOf', str(INPUT_PATH), entry], check=True, stdout=destination)
    else:
        with zipfile.ZipFile(INPUT_PATH) as archive:
            for entry in archive.infolist():
                name = entry.filename.replace('\\', '/')
                target = (DATA_DIR / name).resolve()
                if not target.is_relative_to(DATA_DIR.resolve()):
                    raise ValueError('Unsafe archive path: ' + name)
                if name.lower().endswith(('.rda', '.rdata', '.csv')):
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(entry) as source, target.open('wb') as destination:
                        shutil.copyfileobj(source, destination)
else:
    DATA_DIR = INPUT_PATH if INPUT_PATH.is_dir() else INPUT_PATH.parent
print('Reading tables from', DATA_DIR)

In [ ]:
CLINICAL_TABLES = ['CDR', 'DXSUM', 'PTDEMOG', 'APOERES', 'MMSE', 'ADAS', 'FAQ', 'NEUROBAT', 'NPI']
MRI_TABLE = 'UCSFFSX51'
BIOMARKER_TABLES = {
    'UPENNBIOMK_MASTER': ('DRAWDTE', ['ABETA', 'TAU', 'PTAU']),
    'UPENNBIOMK_ROCHE_ELECSYS': ('EXAMDATE', ['ABETA40', 'ABETA42', 'TAU', 'PTAU']),
    'BLENNOWPLASMANFL': ('EXAMDATE', ['PLASMA_NFL']),
    'UGOTPTAU181': ('EXAMDATE', ['PLASMAPTAU181']),
    'C2N_PRECIVITYAD2_PLASMA': ('EXAMDATE', ['pT217_C2N', 'npT217_C2N', 'AB42_C2N', 'AB40_C2N', 'AB42_AB40_C2N', 'pT217_npT217_C2N']),
    'UPENN_PLASMA_FUJIREBIO_QUANTERIX': ('EXAMDATE', ['pT217_F', 'AB42_F', 'AB40_F', 'AB42_AB40_F', 'pT217_AB42_F', 'NfL_Q', 'GFAP_Q'])}
REFERENCE_TABLES = ['REGISTRY', 'ADNI2_VISITID', 'VISITS', 'DATADIC', 'DERIVED_DATADIC', 'DATA_DOWNLOADED_DATE']
ALL_TABLES = CLINICAL_TABLES + [MRI_TABLE] + list(BIOMARKER_TABLES) + REFERENCE_TABLES

tables, manifest = {}, []
for name in ALL_TABLES:
    matches = [p for p in DATA_DIR.rglob('*') if p.is_file() and p.stem.upper() == name.upper() and p.suffix.lower() in ['.rda', '.rdata', '.csv']]
    if len(matches) != 1:
        raise ValueError(f'{name}: expected exactly one source file, found {len(matches)}.')
    path = matches[0]
    if path.suffix.lower() == '.csv':
        frame = pd.read_csv(path, dtype='string', keep_default_na=False).replace({'': pd.NA, 'NA': pd.NA, '<NA>': pd.NA, 'NaN': pd.NA})
    else:
        objects = pyreadr.read_r(str(path))
        frame = objects.get(name, next(iter(objects.values()))).copy()
    tables[name] = frame.reset_index(drop=True)
    manifest.append({'table': name, 'rows': len(frame), 'columns': len(frame.columns), 'file': path.name,
                     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
manifest = pd.DataFrame(manifest)
manifest.to_csv(OUT / 'input_manifest.csv', index=False)
show(manifest[['table', 'rows', 'columns']], 'Source tables', limit=len(manifest))

## 3. Data Cleaning

Every table is cleaned with the same rules before anything is merged. A patient identifier must be a positive integer, dates must be unambiguous ISO dates inside the study window, rows with an explicit quality-control error are held back, and when one patient has two records on the same measurement date we hold both for review instead of silently picking one. Missing codes are converted only when the ADNI data dictionary documents them for that field, no value is imputed or repaired, and every excluded row keeps its source file, row number and reason, so the cleaning is fully auditable.

Each measurement keeps two dates: the date it was taken (`sample_date`) and a conservative proxy for when it became available in the database (`available_date`). The clean result is one long **observation ledger** with one row per accepted measurement.

### 3.1 Shared cleaning rules

In [ ]:
issues, observations, row_audit, feature_specs = [], [], [], []
dictionary = tables['DATADIC']

def txt(series):
    return series.astype('string').str.strip()

def parse_dates(series):
    """Accept ISO dates and timestamps only; nothing is guessed from ambiguous day/month formats."""
    series = txt(series)
    year = pd.to_numeric(series.str.slice(0, 4), errors='coerce')
    iso = series.str.slice(0, 10).where(series.str.match(r'^\d{4}-\d{2}-\d{2}(?:$|[ T])', na=False) & year.between(1900, 2100))
    return pd.to_datetime(iso, format='%Y-%m-%d', errors='coerce').astype('datetime64[ns]')

def flag(frame, mask, reason, feature='*'):
    """Record the rows matching `mask` as held for review and return the mask."""
    mask = pd.Series(mask, index=frame.index).fillna(False).astype(bool)
    if mask.any():
        held = frame.loc[mask, ['source', 'source_row', 'RID', 'sample_date']].copy()
        held['feature'], held['reason'] = feature, reason
        issues.append(held)
    return mask

def definitions(table, column):
    return dictionary[dictionary.TBLNAME.eq(table) & dictionary.FLDNAME.eq(column)]

def numeric(frame, column):
    """Parse a numeric field; only missing codes documented in DATADIC for this field count as missing."""
    raw = txt(frame[column])
    values = pd.to_numeric(raw, errors='coerce').astype(float)
    codes = ';'.join(definitions(frame['source'].iloc[0], column).CODE.dropna().astype(str))
    sentinels = {float(x) for x in re.findall(r'(?<![\w.])-\d+(?:\.\d+)?(?=\s*(?:=|;|$))', codes)}
    if sentinels:
        flag(frame, values.isin(sentinels), 'documented_missing_code', column)
        values = values.mask(values.isin(sentinels))
    flag(frame, raw.notna() & values.isna() & ~pd.to_numeric(raw, errors='coerce').isin(sentinels), 'missing_or_non_numeric_value', column)
    return values.where(np.isfinite(values))

def prepare(name, date_column, kind='clinical'):
    """Apply the shared identifier, date, QC, duplicate and availability rules to one table."""
    frame = tables[name].copy()
    frame['source'], frame['source_row'] = name, np.arange(1, len(frame) + 1)
    rid = pd.to_numeric(frame.RID, errors='coerce')
    valid_rid = rid.notna() & np.isfinite(rid) & rid.gt(0) & rid.mod(1).eq(0)
    frame['RID'] = rid.where(valid_rid).astype('Int64')
    frame['sample_date'] = parse_dates(frame[date_column])
    bad = flag(frame, ~valid_rid, 'invalid_RID') | flag(frame, frame.sample_date.isna(), 'missing_or_invalid_measurement_date')
    bad |= flag(frame, frame.sample_date.notna() & ~frame.sample_date.between(pd.Timestamp('2004-01-01'), AS_OF_DATE), 'measurement_date_outside_study_window')
    labels = txt(frame['HAS_QC_ERROR']) if 'HAS_QC_ERROR' in frame else pd.Series(pd.NA, index=frame.index, dtype='string')
    known_labels = ['Has QC error', 'Does not have QC error or QC error has been approved']
    bad |= flag(frame, labels.eq('Has QC error'), 'explicit_QC_error')
    bad |= flag(frame, labels.notna() & ~labels.isin(known_labels), 'unknown_QC_label')
    eligible = pd.Series(True, index=frame.index)
    if name == 'UPENNBIOMK_MASTER':                      # keep the source-provided median of the assay batches
        eligible = txt(frame.BATCH).eq('MEDIAN').fillna(False)
        bad |= flag(frame, ~eligible, 'individual_assay_batch_not_selected')
    duplicate = pd.Series(False, index=frame.index)
    duplicate.loc[eligible] = frame.loc[eligible].duplicated(['RID', 'sample_date'], keep=False) & valid_rid.loc[eligible] & frame.loc[eligible, 'sample_date'].notna()
    bad |= flag(frame, duplicate, 'ambiguous_same_patient_measurement_date')
    # Availability proxy: record creation/modification for clinical forms, snapshot update for assays and MRI.
    created = parse_dates(frame.USERDATE) if 'USERDATE' in frame else pd.Series(pd.NaT, index=frame.index, dtype='datetime64[ns]')
    processed = parse_dates(frame.RUNDATE) if 'RUNDATE' in frame else pd.Series(pd.NaT, index=frame.index, dtype='datetime64[ns]')
    if kind in ['assay', 'mri']:
        proxy = parse_dates(frame.update_stamp) if 'update_stamp' in frame else pd.Series(pd.NaT, index=frame.index, dtype='datetime64[ns]')
        bad |= flag(frame, proxy.isna(), 'missing_snapshot_availability_proxy')
    else:
        proxy = created
        if 'USERDATE2' in frame:
            proxy = pd.concat([proxy, parse_dates(frame.USERDATE2)], axis=1).max(axis=1)
        bad |= flag(frame, proxy.isna(), 'missing_record_availability_proxy')
    if kind == 'mri':
        bad |= flag(frame, processed.isna(), 'missing_MRI_processing_date')
    frame['available_date'] = pd.concat([frame.sample_date, proxy, processed], axis=1).max(axis=1)
    bad |= flag(frame, frame.available_date.gt(AS_OF_DATE), 'availability_after_data_cutoff')
    frame['row_usable'] = ~bad
    row_audit.append({'table': name, 'original_rows': len(frame), 'review_rows': int(bad.sum()), 'duplicate_rows': int(duplicate.sum())})
    return frame

def emit(frame, column, feature=None, lo=None, hi=None, unit=None, extra_ok=None, values=None, transform='identity'):
    """Validate one measurement column and append the accepted observations to the ledger."""
    name = frame['source'].iloc[0]
    feature = feature or f'{name}__{column}'
    values = numeric(frame, column) if values is None else values
    valid = values.notna() & np.isfinite(values)
    if lo is not None: valid &= values.ge(lo)
    if hi is not None: valid &= values.le(hi)
    flag(frame, values.notna() & ~valid, 'out_of_range', feature)
    flag(frame, values.isna(), 'missing_value', feature)
    if extra_ok is not None: valid &= extra_ok.fillna(False)
    if unit is None:
        units = definitions(name, column).UNITS.dropna().astype(str)
        units = units[~units.isin(['-4', '0', ''])].unique()
        unit = ' / '.join(units) if len(units) else 'not specified'
    feature_specs.append({'feature': feature, 'source': name, 'source_column': column, 'unit': unit,
                          'lower_bound': lo, 'upper_bound': hi, 'transform': transform})
    take = frame.row_usable & valid
    accepted = frame.loc[take, ['RID', 'source', 'source_row', 'sample_date', 'available_date']].copy()
    for key in ['COLPROT', 'VISCODE']:
        accepted[key] = frame.loc[take, key] if key in frame else pd.NA
    accepted['source_column'], accepted['feature'], accepted['transform'] = column, feature, transform
    accepted['value'] = values.loc[take].astype(float)
    observations.append(accepted)
    return accepted

### 3.2 Clinical scores and diagnosis

CDR-SB is the sum of six box scores, so we accept it only when it is consistent with complete, valid box scores and moves in the expected 0.5-point increments. The diagnosis (CN, MCI, Dementia) is cleaned here as well, but we keep it strictly as metadata: it will select evaluation examples and stratify the split, never feed the model.

In [ ]:
cdr = prepare('CDR', 'VISDATE')
cdrsb = numeric(cdr, 'CDRSB')
boxes = cdr[['CDMEMORY', 'CDORIENT', 'CDJUDGE', 'CDCOMMUN', 'CDHOME', 'CDCARE']].apply(pd.to_numeric, errors='coerce')
boxes_valid = boxes.notna().all(axis=1) & boxes.isin([0, 0.5, 1, 2, 3]).all(axis=1) & boxes.CDCARE.isin([0, 1, 2, 3])
mismatch = boxes_valid & cdrsb.notna() & ~np.isclose(cdrsb, boxes.sum(axis=1), atol=1e-8)
bad_increment = cdrsb.notna() & ~np.isclose(cdrsb * 2, (cdrsb * 2).round(), atol=1e-8)
flag(cdr, mismatch, 'CDRSB_differs_from_box_sum', 'cdrsb')
flag(cdr, bad_increment, 'unusual_CDRSB_increment', 'cdrsb')
emit(cdr, 'CDRSB', 'cdrsb', 0, 18, 'points', ~(mismatch | bad_increment), values=cdrsb)

dx = prepare('DXSUM', 'EXAMDATE')
diagnosis = txt(dx.DIAGNOSIS)
flag(dx, ~diagnosis.isin(['CN', 'MCI', 'Dementia']), 'missing_or_unknown_diagnosis', 'diagnosis_code')
ad_recorded = txt(dx.DXAD).eq('Yes').fillna(False) | txt(dx.DXDDUE).eq("Dementia due to Alzheimer's Disease").fillna(False)
other_cause = txt(dx.DXDDUE).eq('Dementia due to other etiology').fillna(False)
flag(dx, ad_recorded & other_cause, 'conflicting_etiology', 'diagnosis_code')
emit(dx, 'DIAGNOSIS', 'diagnosis_code', 0, 2, 'CN=0; MCI=1; Dementia=2', ~(ad_recorded & other_cause),
     diagnosis.map({'CN': 0, 'MCI': 1, 'Dementia': 2}).astype(float), 'diagnosis_mapping')

SCORE_TABLES = {
    'MMSE': ('VISDATE', [('MMSCORE', 'mmse', 0, 30)]),
    'ADAS': ('VISDATE', [('TOTAL13', 'adas13', 0, 85)]),
    'FAQ': ('VISDATE', [('FAQTOTAL', 'faq', 0, 30)]),
    'NPI': ('EXAMDATE', [('NPITOTAL', 'npi_total', 0, 144)]),
    'NEUROBAT': ('VISDATE', [('LIMMTOTAL', 'logical_memory_immediate', 0, 25), ('AVTOT6', 'ravlt_trial6', 0, 15),
                             ('DSPANFOR', 'digit_span_forward', 0, 12), ('DSPANBAC', 'digit_span_backward', 0, 12),
                             ('BNTTOTAL', 'boston_naming', 0, 30)])}
for name, (date_column, fields) in SCORE_TABLES.items():
    frame = prepare(name, date_column)
    for column, feature, lo, hi in fields:
        emit(frame, column, feature, lo, hi, 'points')

### 3.3 Demographics

Education, sex and age come from the demographics form. ADNI only releases the month and year of birth (sometimes only the year), so we represent the birth date by the 15th of the month, or 1 July for year-only records, and treat age as approximate. The APOE genotype is handled later, when we build the static covariates.

In [ ]:
demographics = prepare('PTDEMOG', 'VISDATE')
emit(demographics, 'PTEDUCAT', 'education', 0, 20, 'years')
emit(demographics, 'PTGENDER', 'female', 0, 1, 'Female=1; Male=0',
     values=txt(demographics.PTGENDER).map({'Female': 1, 'Male': 0}).astype(float), transform='sex_mapping')
birth_text = txt(demographics.PTDOB)
birth = pd.to_datetime(birth_text.where(birth_text.str.match(r'^\d{2}/\d{4}$', na=False)), format='%m/%Y', errors='coerce') + pd.Timedelta(days=14)
birth_year = pd.to_numeric(demographics.PTDOBYY, errors='coerce')
year_only = pd.to_datetime(birth_year.where(birth_year.between(1850, 2026) & birth_year.mod(1).eq(0)).astype('Int64').astype('string') + '-07-01',
                           format='%Y-%m-%d', errors='coerce')
age = (demographics.sample_date - birth.fillna(year_only)).dt.days / 365.25
emit(demographics, 'PTDOB', 'age_at_assessment', 0, 120, 'years', values=age, transform='approximate_DOB_age')

### 3.4 MRI volumes and fluid biomarkers

These channels are not used by the model in this version, but we clean them with the same rigour so that the ledger is complete for future work and so that we can judge their coverage fairly in the next section. MRI regional volumes require an explicit quality-control pass and are expressed as a percentage of intracranial volume to remove differences in head size. Each biomarker assay platform stays a separate channel in its own units, and any assay row carrying a comment is held for review.

In [ ]:
mri = prepare(MRI_TABLE, 'EXAMDATE', 'mri')
icv = numeric(mri, 'ST10CV')
MRI_REGIONS = {'hip_left': ('ST29SV', 'LHIPQC'), 'hip_right': ('ST88SV', 'RHIPQC'),
               'entorhinal_left': ('ST24CV', 'TEMPQC'), 'entorhinal_right': ('ST83CV', 'TEMPQC'),
               'amygdala_left': ('ST12SV', 'CWMQC'), 'amygdala_right': ('ST71SV', 'CWMQC'),
               'ventricle_left': ('ST37SV', 'VENTQC'), 'ventricle_right': ('ST96SV', 'VENTQC')}
for feature, (column, region_qc) in MRI_REGIONS.items():
    volume = numeric(mri, column)
    qc_pass = txt(mri.OVERALLQC).eq('Pass').fillna(False) & txt(mri[region_qc]).eq('Pass').fillna(False)
    plausible = icv.gt(0) & volume.gt(0) & volume.lt(icv)
    flag(mri, ~qc_pass, 'MRI_QC_not_explicitly_passed', feature)
    flag(mri, ~plausible, 'invalid_regional_volume_or_ICV', feature)
    emit(mri, column, feature, 0, 100, '% ICV', qc_pass & plausible, 100 * volume / icv, '100_times_volume_divided_by_ICV')

for name, (date_column, fields) in BIOMARKER_TABLES.items():
    frame = prepare(name, date_column, 'assay')
    has_comment = pd.Series(False, index=frame.index)
    for column in [c for c in frame.columns if c.lower() in ['comment', 'comments']]:
        has_comment |= txt(frame[column]).fillna('').ne('')
    flag(frame, has_comment, 'assay_comment_requires_review')
    for column in fields:
        values = numeric(frame, column)
        flag(frame, values.notna() & ~values.gt(0), 'nonpositive_assay_result', column)
        unit = 'ratio' if column in ['AB42_AB40_C2N', 'AB42_AB40_F', 'pT217_AB42_F'] else ('percent' if column == 'pT217_npT217_C2N' else None)
        emit(frame, column, unit=unit, extra_ok=~has_comment & values.gt(0), values=values)

### 3.5 The observation ledger

We now stack every accepted measurement into one long ledger. If two observations compete for the same patient, feature, measurement date and availability date, both are held for review rather than averaged. We then verify that every ledger row still points at its original source row, and recompute a sample of values, including every derived transform, directly from the raw tables.

In [ ]:
ledger = pd.concat(observations, ignore_index=True)
ledger['RID'] = ledger.RID.astype('int64')
collision = ledger.duplicated(['RID', 'available_date', 'sample_date', 'feature'], keep=False)
ledger = ledger[~collision].sort_values(['RID', 'available_date', 'sample_date', 'feature']).reset_index(drop=True)
ledger['observation_id'] = ledger.source + ':' + ledger.source_row.astype(str) + ':' + ledger.source_column
assert ledger.observation_id.is_unique

review = pd.concat(issues, ignore_index=True).drop_duplicates()
feature_dictionary = pd.DataFrame(feature_specs).drop_duplicates('feature')
ledger.to_csv(OUT / 'observations_long.csv', index=False, date_format='%Y-%m-%d')
review.to_csv(OUT / 'review_reasons.csv', index=False, date_format='%Y-%m-%d')
feature_dictionary.to_csv(OUT / 'feature_dictionary.csv', index=False)

accepted = ledger.groupby('source').agg(accepted_rows=('source_row', 'nunique'), observations=('value', 'size'), patients=('RID', 'nunique'))
cleaning_summary = pd.DataFrame(row_audit).merge(accepted, left_on='table', right_index=True, how='left').fillna(0)
cleaning_summary.to_csv(OUT / 'cleaning_summary.csv', index=False)
show(cleaning_summary, 'Cleaning summary per source table', limit=len(cleaning_summary))
note(f'{len(ledger):,} accepted observations of {ledger.feature.nunique()} features for {ledger.RID.nunique():,} patients; '
     f'{int(collision.sum())} colliding observations and {len(review):,} review entries saved.')

In [ ]:
# Every accepted observation must point at an existing source row with the same RID.
for name, group in ledger.groupby('source'):
    original = tables[name].iloc[group.source_row.to_numpy() - 1]
    assert np.array_equal(pd.to_numeric(original.RID).to_numpy(), group.RID.to_numpy()), name

# Recompute three observations per feature from the raw tables, including every derived transform.
spot_checks = []
for row in ledger.groupby('feature', group_keys=False).head(3).itertuples():
    original = tables[row.source].iloc[row.source_row - 1]
    raw = original[row.source_column]
    if row.transform == 'identity': expected = float(raw)
    elif row.transform == '100_times_volume_divided_by_ICV': expected = 100 * float(raw) / float(original.ST10CV)
    elif row.transform == 'sex_mapping': expected = {'Female': 1, 'Male': 0}[str(raw)]
    elif row.transform == 'diagnosis_mapping': expected = {'CN': 0, 'MCI': 1, 'Dementia': 2}[str(raw)]
    elif row.transform == 'approximate_DOB_age':
        text = str(original.PTDOB)
        born = pd.to_datetime(text, format='%m/%Y') + pd.Timedelta(days=14) if re.fullmatch(r'\d{2}/\d{4}', text) else pd.Timestamp(year=int(float(original.PTDOBYY)), month=7, day=1)
        expected = (row.sample_date - born).days / 365.25
    else: raise ValueError('Unknown transform: ' + row.transform)
    spot_checks.append({'observation_id': row.observation_id, 'source_value': str(raw), 'expected': expected, 'exported': row.value,
                        'matches': bool(np.isclose(expected, row.value, rtol=1e-10, atol=1e-10))})
spot_checks = pd.DataFrame(spot_checks)
assert spot_checks.matches.all()
note(f'Source identifiers verified for {ledger.source.nunique()} tables; {len(spot_checks)} recomputed values match the ledger.')

## 4. Building the Patient Visit Table

With a clean ledger in hand, we decide what the model will see. We use the four core clinical scores as time-varying channels, CDR-SB, MMSE, ADAS-13 and FAQ, because they are recorded at almost every visit for thousands of patients and together cover global severity, cognition and daily function. Four static covariates (baseline age, sex, education and APOE ε4 allele count) initialise the hidden state. MRI and fluid biomarkers reach far fewer patients on sparser schedules, so we keep them in the ledger for a later version. The diagnosis remains metadata.

A visit is one patient on one measurement date. Clinical scores are known in clinic on the day they are taken, so we use the measurement date as the clock and treat the database availability proxy as a sensitivity diagnostic only.

In [ ]:
ledger['measurement_date'] = ledger['sample_date']
ledger['proxy_delay_days'] = (ledger.available_date - ledger.sample_date).dt.days
STATIC_SOURCE_FEATURES = ['age_at_assessment', 'female', 'education']

def feature_group(feature, source):
    if feature in CHANNELS: return 'model channel'
    if feature in STATIC_SOURCE_FEATURES: return 'static covariate'
    if feature == 'diagnosis_code': return 'metadata only (never an input)'
    if source == MRI_TABLE: return 'MRI, kept for future work'
    if source in BIOMARKER_TABLES: return 'biomarker, kept for future work'
    return 'additional clinical score, not used'

viability = ledger.groupby('feature').agg(source=('source', 'first'), observations=('value', 'size'), patients=('RID', 'nunique')).reset_index()
viability['role'] = [feature_group(f, s) for f, s in zip(viability.feature, viability.source)]
viability = viability.sort_values(['role', 'observations'], ascending=[True, False]).reset_index(drop=True)
viability.to_csv(OUT / 'channel_viability.csv', index=False)
show(viability, 'Coverage of every cleaned feature and its role in the model', limit=len(viability))

In [ ]:
clinical = ledger[ledger.feature.isin(CHANNELS)]
availability = clinical.groupby('COLPROT').agg(observations=('value', 'size'),
    share_proxy_within_30_days=('proxy_delay_days', lambda d: (d <= 30).mean()),
    median_proxy_delay_days=('proxy_delay_days', 'median')).reset_index()
show(availability, 'Database availability proxy for the clinical scores, by protocol (sensitivity diagnostic only)')

obs = ledger[ledger.feature.isin(CHANNELS)].sort_values(['RID', 'measurement_date', 'feature']).reset_index(drop=True)

def build_visits(observations):
    """One row per (patient, measurement date) with a value and a 0/1 mask per channel and a clock in years."""
    key = ['RID', 'measurement_date']
    assert not observations.duplicated(key + ['feature']).any(), 'a channel has two values for one patient and date'
    values = observations.pivot(index=key, columns='feature', values='value').reindex(columns=CHANNELS)
    ordered = observations.assign(from_cdr=observations.feature.eq('cdrsb')).sort_values(key + ['from_cdr'], ascending=[True, True, False])
    meta = ordered.groupby(key).agg(COLPROT=('COLPROT', 'first'), VISCODE=('VISCODE', 'first'),
                                    META_proxy_delay_days=('proxy_delay_days', 'max'))
    visits = values.join(meta).reset_index()
    visits = visits[visits[CHANNELS].notna().any(axis=1)].sort_values(key).reset_index(drop=True)
    for channel in CHANNELS:
        visits['mask_' + channel] = visits[channel].notna().astype(int)
    visits['t_years'] = (visits.measurement_date - visits.groupby('RID').measurement_date.transform('min')).dt.days / 365.25
    visits['visit_index'] = visits.groupby('RID').cumcount()
    return visits

visits_all = build_visits(obs)
note(f'{len(obs):,} channel observations form {len(visits_all):,} visits for {visits_all.RID.nunique():,} patients.')

### 4.1 Static covariates

We take the first recorded sex and education per patient and shift the earliest recorded age to the date of the first visit. The APOE genotype never changes, so we keep it for every patient with a valid genotype; the `APUSABLE` sample flag exists only for ADNI1 records and would otherwise remove APOE for every later protocol. Each static covariate carries a mask so that an unknown value (for example an untested APOE) is represented honestly rather than guessed.

In [ ]:
demographic_obs = ledger[ledger.feature.isin(STATIC_SOURCE_FEATURES)].sort_values(['RID', 'sample_date'])
first_values = demographic_obs.groupby(['RID', 'feature']).agg(value=('value', 'first'), sample_date=('sample_date', 'first'))
static_all = first_values['value'].unstack('feature').reindex(columns=['female', 'education', 'age_at_assessment'])
age_date = first_values['sample_date'].unstack('feature')['age_at_assessment']
static_all = static_all.join(visits_all.groupby('RID').measurement_date.min().rename('first_visit'), how='right')
static_all['baseline_age'] = static_all.age_at_assessment + (static_all.first_visit - age_date.reindex(static_all.index)).dt.days / 365.25

apoe_raw = tables['APOERES'].copy()
apoe_raw['RID'] = pd.to_numeric(apoe_raw.RID, errors='coerce')
genotype = txt(apoe_raw.GENOTYPE)
valid_genotype = genotype.str.fullmatch(r'[234]/[234]', na=False).astype(bool) & apoe_raw.RID.notna()
has_sample_record = txt(apoe_raw.APUSABLE).eq('Yes').fillna(False).astype(bool)
apoe = pd.DataFrame({'RID': apoe_raw.RID[valid_genotype].astype('int64'),
                     'apoe4_count': genotype[valid_genotype].str.count('4').astype(float),
                     'META_apoe_source': np.where(has_sample_record[valid_genotype], 'adni1_sample_record', 'genotype_only')})
assert apoe.groupby('RID').apoe4_count.nunique().max() == 1, 'conflicting APOE genotypes for one patient'
apoe = apoe.sort_values(['RID', 'META_apoe_source']).drop_duplicates('RID').set_index('RID')
static_all = static_all.join(apoe)
static_all['META_apoe_source'] = static_all.META_apoe_source.fillna('no_genotype')
for column in STATICS:
    static_all['mask_' + column] = static_all[column].notna().astype(int)
static_all = static_all.reset_index()

first_protocol = visits_all.sort_values('measurement_date').groupby('RID').COLPROT.first()
apoe_table = static_all.assign(protocol=static_all.RID.map(first_protocol)).groupby('protocol').agg(
    patients=('RID', 'size'), with_apoe_genotype=('mask_apoe4_count', 'sum'),
    with_adni1_sample_record=('META_apoe_source', lambda s: (s == 'adni1_sample_record').sum())).reset_index()
apoe_table['share_with_genotype'] = apoe_table.with_apoe_genotype / apoe_table.patients
show(apoe_table, 'APOE genotype availability by first protocol')

### 4.2 Diagnosis at each visit

To know which visits belong to MCI patients, we attach a diagnosis to every visit: first by the exact visit key (patient, protocol, visit code) when the two dates are within 90 days, otherwise the nearest diagnosis within 90 days, otherwise "unknown".

In [ ]:
diagnoses = ledger.loc[ledger.feature == 'diagnosis_code', ['RID', 'sample_date', 'COLPROT', 'VISCODE', 'value']]
diagnoses = diagnoses.rename(columns={'sample_date': 'dx_date', 'value': 'dx_code'}).sort_values('dx_date')
assert not diagnoses.duplicated(['RID', 'COLPROT', 'VISCODE']).any(), 'two diagnoses for one visit key'
DX_LABEL = {0.0: 'CN', 1.0: 'MCI', 2.0: 'Dementia'}

def link_diagnosis(visits):
    linked = visits.merge(diagnoses, on=['RID', 'COLPROT', 'VISCODE'], how='left')
    gap = (linked.dx_date - linked.measurement_date).dt.days.abs()
    by_key = linked.dx_date.notna() & (gap <= CFG.MAX_GAP_DAYS)
    linked.loc[~by_key, ['dx_date', 'dx_code']] = np.nan
    linked['dx_source'] = np.where(by_key, 'visit_key', 'unknown')
    pending = linked.loc[~by_key, ['RID', 'measurement_date']].reset_index().sort_values('measurement_date')
    nearest = pd.merge_asof(pending, diagnoses[['RID', 'dx_date', 'dx_code']], left_on='measurement_date', right_on='dx_date',
                            by='RID', tolerance=pd.Timedelta(days=CFG.MAX_GAP_DAYS), direction='nearest').set_index('index')
    found = nearest[nearest.dx_date.notna()]
    linked.loc[found.index, 'dx_date'], linked.loc[found.index, 'dx_code'] = found.dx_date, found.dx_code
    linked.loc[found.index, 'dx_source'] = 'nearest_date'
    linked['dx'] = linked.dx_code.map(DX_LABEL).fillna('unknown')
    return linked.drop(columns=['dx_date', 'dx_code'])

visits_all = link_diagnosis(visits_all)
dx_table = pd.crosstab(visits_all.dx, visits_all.dx_source).rename_axis(columns=None).reset_index()
show(dx_table, 'Diagnosis at visit (rows) by how it was linked (columns)')

### 4.3 Modeling population

A patient with a single visit carries no trajectory to learn from, so the modeling population contains every patient with at least two visits. Visits without CDR-SB but with another score stay in, because they still inform the hidden state. Finally, we flag implausible CDR-SB jumps (a rise of 4 or more points within six months, or an improvement of 3 or more points) without removing them: they may be genuine, and deleting them would make the model look better than it is.

In [ ]:
visits_per_patient = visits_all.groupby('RID').RID.transform('size')
population = visits_all[visits_per_patient >= CFG.MIN_VISITS].reset_index(drop=True)
obs_population = obs[obs.RID.isin(population.RID)].copy()

cdr_only = population[population.mask_cdrsb == 1].sort_values(['RID', 't_years']).copy()
cdr_only['previous_cdrsb'] = cdr_only.groupby('RID').cdrsb.shift()
cdr_only['delta'] = cdr_only.cdrsb - cdr_only.previous_cdrsb
cdr_only['days_between'] = (cdr_only.measurement_date - cdr_only.groupby('RID').measurement_date.shift()).dt.days
jump_up = (cdr_only.delta >= CFG.JUMP_UP) & (cdr_only.days_between <= CFG.JUMP_DAYS)
jump_down = cdr_only.delta <= -CFG.JUMP_DOWN
flagged_jumps = cdr_only.loc[jump_up | jump_down, ['RID', 'measurement_date', 'days_between', 'previous_cdrsb', 'cdrsb', 'delta']].copy()
flagged_jumps['flag_type'] = np.where(jump_up[jump_up | jump_down], 'rapid_increase', 'improvement')
flagged_jumps.to_csv(OUT / 'flagged_jumps.csv', index=False, date_format='%Y-%m-%d')

baseline = population.sort_values('measurement_date').groupby('RID').agg(
    META_baseline_protocol=('COLPROT', 'first'), META_baseline_dx=('dx', 'first'), n_visits=('RID', 'size')).reset_index()
static_population = static_all[static_all.RID.isin(population.RID)].merge(baseline, on='RID', how='left')

attrition = pd.DataFrame([
    ('Patients in the cleaned ledger', ledger.RID.nunique(), np.nan),
    ('Patients with at least one visit in the four channels', visits_all.RID.nunique(), len(visits_all)),
    (f'Patients with at least {CFG.MIN_VISITS} visits (modeling population)', population.RID.nunique(), len(population))],
    columns=['step', 'patients', 'visits']).astype({'visits': 'Int64'})
show(attrition, 'Patients and visits at each step')
note(f'{len(flagged_jumps)} CDR-SB observations flagged ({int(jump_up.sum())} rapid increases, {int(jump_down.sum())} improvements) and kept in the data.')

## 5. Exploratory Data Analysis

Before modeling, we look at who is in the population and how the data behaves over time. Three questions matter most for the model we are about to build: how the cohort is composed, how irregular the visit schedule is, and how the four scores are distributed and change along a patient's trajectory.

In [ ]:
protocol_order = [p for p in ['ADNI1', 'ADNIGO', 'ADNI2', 'ADNI3', 'ADNI4'] if p in set(baseline.META_baseline_protocol)]
dx_order = [d for d in ['CN', 'MCI', 'Dementia', 'unknown'] if d in set(baseline.META_baseline_dx)]
composition = pd.crosstab(baseline.META_baseline_protocol, baseline.META_baseline_dx).reindex(index=protocol_order, columns=dx_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(8, 3.4))
bottom = np.zeros(len(composition))
for dx_label, color in zip(dx_order, PALETTE):
    ax.bar(composition.index, composition[dx_label], bottom=bottom, color=color, label=dx_label, edgecolor='white')
    bottom += composition[dx_label].to_numpy()
ax.set_ylabel('Patients'); ax.legend(frameon=False, fontsize=9, title='Diagnosis at first visit', title_fontsize=9, loc='upper left', bbox_to_anchor=(1.01, 1))
style_axis(ax, 'Modeling population by enrolment protocol and baseline diagnosis')
finish(fig, 'eda_cohort_composition')
show(composition.assign(total=composition.sum(axis=1)).reset_index(), 'Patients by protocol and baseline diagnosis')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
visit_counts = baseline.n_visits.clip(upper=15).value_counts().sort_index()
axes[0].bar(visit_counts.index, visit_counts.values, color=MID, edgecolor=PURPLE)
axes[0].set_xlabel('Visits per patient (15 = 15 or more)'); axes[0].set_ylabel('Patients')
style_axis(axes[0], 'Follow-up length')
gaps_months = population.groupby('RID').t_years.diff().dropna() * 12
axes[1].hist(gaps_months.clip(upper=36), bins=36, color=MID, edgecolor='white')
axes[1].set_xlabel('Months between consecutive visits (36 = 36 or more)'); axes[1].set_ylabel('Transitions')
style_axis(axes[1], 'Irregular visit spacing the model has to bridge')
finish(fig, 'eda_visits_and_gaps')
note(f'Median {baseline.n_visits.median():.0f} visits per patient over a median follow-up of '
     f'{population.groupby("RID").t_years.max().median():.1f} years; median gap between visits {gaps_months.median():.1f} months.')

In [ ]:
RANGES = {'cdrsb': (0, 18), 'mmse': (0, 30), 'adas13': (0, 85), 'faq': (0, 30)}
LABELS = {'cdrsb': 'CDR-SB (0–18, higher = worse)', 'mmse': 'MMSE (0–30, higher = better)',
          'adas13': 'ADAS-13 (0–85, higher = worse)', 'faq': 'FAQ (0–30, higher = worse)'}
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, channel in zip(axes.ravel(), CHANNELS):
    observed = population.loc[population['mask_' + channel] == 1, channel]
    lo, hi = RANGES[channel]
    ax.hist(observed, bins=np.linspace(lo, hi, 37), color=MID, edgecolor='white')
    ax.set_xlabel(LABELS[channel]); ax.set_ylabel('Observations')
    style_axis(ax, f'{channel}: observed at {observed.size / len(population):.0%} of visits')
finish(fig, 'eda_score_distributions')
score_stats = pd.DataFrame([{'channel': c, 'observed_visits': int(population['mask_' + c].sum()),
                             'share_of_visits': population['mask_' + c].mean(),
                             'mean': population[c].mean(), 'median': population[c].median(), 'std': population[c].std(),
                             'min': population[c].min(), 'max': population[c].max()} for c in CHANNELS])
show(score_stats, 'Summary statistics of the four channels in the modeling population')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
mean_curve = population[population.mask_cdrsb == 1].assign(year=lambda d: d.t_years.round())
mean_curve = mean_curve.merge(baseline[['RID', 'META_baseline_dx']], on='RID')
for dx_label, color in zip([d for d in dx_order if d != 'unknown'], PALETTE):
    curve = mean_curve[(mean_curve.META_baseline_dx == dx_label) & (mean_curve.year <= 8)].groupby('year').cdrsb.agg(['mean', 'size'])
    curve = curve[curve['size'] >= 30]
    axes[0].plot(curve.index, curve['mean'], marker='o', markersize=4, linewidth=2, color=color, label=dx_label)
axes[0].set_xlabel('Years since first visit'); axes[0].set_ylabel('Mean CDR-SB'); axes[0].legend(frameon=False, fontsize=9, title='Baseline diagnosis', title_fontsize=9)
style_axis(axes[0], 'Average CDR-SB course by baseline diagnosis')

mci_patients = baseline[baseline.META_baseline_dx == 'MCI'].RID
mci_cdr = population[population.RID.isin(mci_patients) & (population.mask_cdrsb == 1)].sort_values(['RID', 't_years'])
eligible = mci_cdr.groupby('RID').agg(n=('cdrsb', 'size'), last=('cdrsb', 'last'))
eligible = eligible[eligible.n >= 6].sort_values(['last', 'n'])
for i, rid in enumerate(eligible.index[np.linspace(0, len(eligible) - 1, 6).astype(int)]):
    patient = mci_cdr[mci_cdr.RID == rid]
    axes[1].plot(patient.t_years, patient.cdrsb, marker='o', markersize=4, linewidth=1.4, color=PALETTE[i % len(PALETTE)], label=f'RID {rid}')
axes[1].set_xlabel('Years since first visit'); axes[1].set_ylabel('CDR-SB'); axes[1].legend(fontsize=8, frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1))
style_axis(axes[1], 'Individual MCI trajectories: the same diagnosis, very different courses')
finish(fig, 'eda_cdrsb_trajectories')

## 6. Preprocessing: Split, Normalization and Evaluation Examples

Everything that could leak information is handled here, in this order. We first assign every patient to one of four groups by patient identifier, stratified by baseline diagnosis and follow-up length: 70% for training, 10% for validation (early stopping), 10% for calibrating the uncertainty, and 10% for a test set that is touched once at the end. Only then do we standardise the channels and the continuous covariates with statistics from training patients, and build the evaluation examples. Missing entries become value 0 with mask 0 after normalization; nothing is imputed.

In [ ]:
def assign_splits(patients, seed):
    """Patient-level split, stratified by baseline diagnosis and visit-count bin."""
    bins = CFG.VISIT_BINS + [np.inf]
    bin_labels = [f'{lo}' if hi == lo + 1 else (f'{lo}+' if hi == np.inf else f'{lo}-{int(hi) - 1}') for lo, hi in zip(bins[:-1], bins[1:])]
    stratum = patients.META_baseline_dx + ' | ' + pd.cut(patients.n_visits, bins=bins, labels=bin_labels, right=False).astype(str)
    rng = np.random.default_rng(seed)
    cuts = np.cumsum([CFG.SPLIT_RATIOS[s] for s in SPLITS])
    assignment = {}
    for _, group in patients.assign(stratum=stratum).groupby('stratum', sort=True):
        rids = rng.permutation(np.sort(group.RID.to_numpy()))
        bounds = np.round(cuts * len(rids)).astype(int)
        for name, lo, hi in zip(SPLITS, np.r_[0, bounds[:-1]], bounds):
            assignment.update({int(rid): name for rid in rids[lo:hi]})
    return pd.Series(assignment, name='split')

split = assign_splits(static_population, CFG.SEED)
static_population['split'] = static_population.RID.map(split)
population['split'] = population.RID.map(split)
assert population.split.notna().all() and static_population.split.notna().all()

split_table = static_population.groupby('split').agg(patients=('RID', 'size'), visits=('n_visits', 'sum'), median_visits=('n_visits', 'median')).reindex(SPLITS)
split_table = split_table.join(pd.crosstab(static_population.split, static_population.META_baseline_dx).add_prefix('baseline_')).rename_axis(columns=None).reset_index()
show(split_table, 'Patient-level split (baseline diagnosis is used for stratification only)')

In [ ]:
CONTINUOUS_STATIC = [s for s in STATICS if s in ('baseline_age', 'education')]

def fit_normalization(train_visits, train_static):
    params = {'channels': {}, 'static': {}}
    for channel in CHANNELS:
        observed = train_visits.loc[train_visits['mask_' + channel] == 1, channel]
        params['channels'][channel] = {'mean': float(observed.mean()), 'std': float(observed.std(ddof=0)) or 1.0, 'n_train': int(observed.size)}
    for feature in CONTINUOUS_STATIC:
        observed = train_static.loc[train_static['mask_' + feature] == 1, feature]
        params['static'][feature] = {'mean': float(observed.mean()), 'std': float(observed.std(ddof=0)) or 1.0, 'n_train': int(observed.size)}
    return params

def apply_normalization(visits, static_table, params):
    sequences = visits[['RID', 'visit_index', 't_years', 'measurement_date']].copy()
    for channel in CHANNELS:
        p = params['channels'][channel]
        sequences['value_' + channel] = ((visits[channel] - p['mean']) / p['std']).where(visits['mask_' + channel] == 1, 0.0)
        sequences['mask_' + channel] = visits['mask_' + channel].astype(int)
    statics = static_table[['RID']].copy()
    for feature in STATICS:
        p = params['static'].get(feature, {'mean': 0.0, 'std': 1.0})
        statics[feature] = ((static_table[feature] - p['mean']) / p['std']).where(static_table['mask_' + feature] == 1, 0.0)
        statics['mask_' + feature] = static_table['mask_' + feature].astype(int)
    for column in ['META_baseline_protocol', 'META_baseline_dx', 'META_apoe_source']:
        statics[column] = static_table[column].to_numpy()
    return sequences, statics

norm = fit_normalization(population[population.split == 'train'], static_population[static_population.split == 'train'])
sequences_all, static_norm = apply_normalization(population, static_population, norm)
sequences_all['split'], static_norm['split'] = population.split.to_numpy(), static_population.split.to_numpy()
norm_table = pd.DataFrame([{'variable': k, 'kind': 'channel', **v} for k, v in norm['channels'].items()] +
                          [{'variable': k, 'kind': 'static', **v} for k, v in norm['static'].items()])
show(norm_table, 'Normalization parameters fitted on training patients only (female and apoe4_count stay as 0/1 and 0/1/2)')

### 6.1 Evaluation examples

Training uses every visit of every patient. To report forecasts at fixed horizons we additionally define evaluation examples: an **index visit** is any visit with an observed CDR-SB, an MCI diagnosis, and at least two CDR-SB observations so far; its **target** is the CDR-SB observation closest to the index date plus the horizon, inside a tolerance window and strictly after the index. The model will only read visits up to and including the index visit. Because an example needs a future observation, patients lost to follow-up are under-represented, and we quantify that.

In [ ]:
def build_examples(visits, params):
    ordered = visits.sort_values(['RID', 't_years']).copy()
    ordered['cdr_observations_so_far'] = ordered.groupby('RID').mask_cdrsb.cumsum()
    index_visits = ordered[(ordered.mask_cdrsb == 1) & (ordered.dx == CFG.EVAL_DX) & (ordered.cdr_observations_so_far >= CFG.MIN_CDR_OBS_EVAL)]
    targets = ordered.loc[ordered.mask_cdrsb == 1, ['RID', 't_years', 'measurement_date', 'cdrsb']].rename(
        columns={'t_years': 'target_time', 'measurement_date': 'target_date', 'cdrsb': 'target_cdrsb_raw'})
    protocol_of = static_population.set_index('RID').META_baseline_protocol
    baseline_dx_of = static_population.set_index('RID').META_baseline_dx
    frames, coverage_rows = [], []
    for horizon, window in CFG.HORIZONS.items():
        pairs = index_visits.merge(targets, on='RID')
        pairs = pairs[pairs.target_time > pairs.t_years]
        pairs['delta_years'] = pairs.target_time - pairs.t_years
        pairs['distance_to_horizon'] = (pairs.delta_years - horizon / 12).abs()
        pairs = pairs[pairs.distance_to_horizon <= window / 12].sort_values(['RID', 't_years', 'distance_to_horizon']).drop_duplicates(['RID', 't_years'])
        frames.append(pd.DataFrame({
            'RID': pairs.RID, 'split': pairs.split, 'horizon_months': horizon,
            'index_visit_index': pairs.visit_index, 'n_input_visits': pairs.visit_index + 1,
            'index_time': pairs.t_years, 'index_date': pairs.measurement_date, 'index_cdrsb_raw': pairs.cdrsb,
            'target_time': pairs.target_time, 'target_date': pairs.target_date, 'delta_years': pairs.delta_years,
            'target_cdrsb_raw': pairs.target_cdrsb_raw, 'dx_at_index': pairs.dx,
            'META_baseline_protocol': pairs.RID.map(protocol_of), 'META_baseline_dx': pairs.RID.map(baseline_dx_of)}))
        coverage_rows.append({'horizon_months': horizon, 'window_months': window, 'eligible_index_visits': len(index_visits),
                              'with_target_in_window': len(pairs), 'share_with_target': len(pairs) / max(len(index_visits), 1),
                              'patients_with_examples': pairs.RID.nunique()})
    return pd.concat(frames, ignore_index=True), pd.DataFrame(coverage_rows)

examples_all, selection_bias = build_examples(population, norm)
example_counts = examples_all.pivot_table(index='split', columns='horizon_months', values='RID', aggfunc=['size', 'nunique']).reindex(SPLITS)
example_counts.columns = [f'{"examples" if a == "size" else "patients"}_{h}m' for a, h in example_counts.columns]
show(example_counts.reset_index(), 'Evaluation examples and patients per split and horizon')
show(selection_bias, 'Share of eligible MCI index visits with a target in the window (patients lost to follow-up are under-represented)')

### 6.2 Dataset and batching

Each patient becomes one item: visit times in years, a value matrix, a mask matrix and the static covariates. Patients of different lengths are padded into a batch; padded steps repeat the patient's last time and carry mask 0, so the ODE never integrates backwards and the padding never updates the state.

In [ ]:
VALUE_COLS, MASK_COLS = ['value_' + c for c in CHANNELS], ['mask_' + c for c in CHANNELS]
STATIC_MASK_COLS = ['mask_' + s for s in STATICS]

class PatientSequenceDataset(Dataset):
    """One patient per item, built from the normalized sequence and static tables."""
    def __init__(self, sequences, static):
        static = static.set_index('RID')
        self.items = []
        for rid, patient in sequences.sort_values(['RID', 't_years']).groupby('RID', sort=True):
            self.items.append({'rid': int(rid),
                               'times': patient.t_years.to_numpy(dtype=np.float32),
                               'values': patient[VALUE_COLS].to_numpy(dtype=np.float32),
                               'masks': patient[MASK_COLS].to_numpy(dtype=np.float32),
                               'static': static.loc[rid, STATICS].to_numpy(dtype=np.float32),
                               'static_mask': static.loc[rid, STATIC_MASK_COLS].to_numpy(dtype=np.float32)})
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]

def collate_fn(batch):
    """Pad variable-length patients into one batch of tensors."""
    lengths = np.array([len(item['times']) for item in batch], dtype=np.int64)
    batch_size, max_len, n_channels = len(batch), int(lengths.max()), batch[0]['values'].shape[1]
    times = np.zeros((batch_size, max_len), dtype=np.float32)
    values = np.zeros((batch_size, max_len, n_channels), dtype=np.float32)
    masks = np.zeros((batch_size, max_len, n_channels), dtype=np.float32)
    for i, item in enumerate(batch):
        n = lengths[i]
        times[i, :n], times[i, n:] = item['times'], item['times'][-1]
        values[i, :n], masks[i, :n] = item['values'], item['masks']
    out = {'rid': np.array([item['rid'] for item in batch], dtype=np.int64), 'times': times, 'values': values, 'masks': masks,
           'lengths': lengths, 'static': np.stack([item['static'] for item in batch]), 'static_mask': np.stack([item['static_mask'] for item in batch])}
    return {key: torch.as_tensor(value) for key, value in out.items()}

sequences = {s: sequences_all[sequences_all.split == s].drop(columns='split').reset_index(drop=True) for s in SPLITS}
statics = {s: static_norm[static_norm.split == s].drop(columns='split').reset_index(drop=True) for s in SPLITS}
examples = {s: examples_all[examples_all.split == s].drop(columns='split').reset_index(drop=True) for s in SPLITS}
datasets = {s: PatientSequenceDataset(sequences[s], statics[s]) for s in SPLITS}
loaders = {s: DataLoader(datasets[s], batch_size=CFG.BATCH_SIZE, shuffle=(s == 'train'), collate_fn=collate_fn) for s in SPLITS}

batch = collate_fn([datasets['train'][i] for i in range(4)])
show(pd.DataFrame([{'tensor': k, 'shape': str(tuple(v.shape)), 'dtype': str(v.dtype).replace('torch.', '')} for k, v in batch.items()]),
     'One batch of four training patients')

### 6.3 Leakage checks

Before training we verify, programmatically, that the split is disjoint, that model inputs for every evaluation example contain nothing after the index visit, that the normalization depends on training patients only, and that no metadata column can reach the model.

In [ ]:
checks = []
def check(label, condition):
    checks.append({'check': label, 'result': 'PASS' if bool(condition) else 'FAIL'})
    assert condition, 'Check failed: ' + label

check('No patient appears in more than one split', static_norm.groupby('RID').split.nunique().max() == 1)
check('Visit times strictly increase within every patient',
      (sequences_all.groupby('RID').t_years.diff().dropna() > 0).all() and not sequences_all.duplicated(['RID', 't_years']).any())
times_by_rid = {rid: g.t_years.to_numpy() for rid, g in sequences_all.groupby('RID')}
check('Every example: all input times <= index time < target time', all(
    times_by_rid[e.RID][:e.n_input_visits].max() <= e.index_time < e.target_time
    and times_by_rid[e.RID][e.n_input_visits:].min(initial=np.inf) > e.index_time for e in examples_all.itertuples()))

prefix_ok = True
for e in examples_all.sample(200, random_state=CFG.SEED).itertuples():
    truncated = obs_population[(obs_population.RID == e.RID) & (obs_population.measurement_date <= e.index_date)]
    rebuilt, _ = apply_normalization(build_visits(truncated), static_population[static_population.RID == e.RID], norm)
    original = sequences_all[(sequences_all.RID == e.RID) & (sequences_all.t_years <= e.index_time)]
    prefix_ok &= np.allclose(rebuilt[['t_years'] + VALUE_COLS + MASK_COLS].to_numpy(dtype=float), original[['t_years'] + VALUE_COLS + MASK_COLS].to_numpy(dtype=float))
check('Model inputs rebuilt from the ledger truncated at the index date are identical (200 random examples)', prefix_ok)

altered, altered_static = population.copy(), static_population.copy()
altered.loc[altered.split != 'train', CHANNELS] = altered.loc[altered.split != 'train', CHANNELS] * 10 + 5
altered_static.loc[altered_static.split != 'train', CONTINUOUS_STATIC] = altered_static.loc[altered_static.split != 'train', CONTINUOUS_STATIC] * 10 + 5
check('Normalization parameters are unchanged when val/calib/test values are altered',
      fit_normalization(altered[altered.split == 'train'], altered_static[altered_static.split == 'train']) == norm)
check('No NaN in model inputs and masks match the observed values',
      not sequences_all[['t_years'] + VALUE_COLS + MASK_COLS].isna().any().any() and not static_norm[STATICS + STATIC_MASK_COLS].isna().any().any()
      and all((sequences_all.loc[sequences_all['mask_' + c] == 0, 'value_' + c] == 0).all() for c in CHANNELS)
      and all((population[c].notna().to_numpy() == (population['mask_' + c] == 1).to_numpy()).all() for c in CHANNELS))
check('Inverse-transformed channel values lie inside their valid ranges', all(
    (sequences_all.loc[sequences_all['mask_' + c] == 1, 'value_' + c] * norm['channels'][c]['std'] + norm['channels'][c]['mean']).between(RANGES[c][0] - 1e-6, RANGES[c][1] + 1e-6).all()
    for c in CHANNELS))
forbidden = ['dx', 'diagnosis', 'colprot', 'protocol', 'apoe_source', 'meta_']
check('Diagnosis, protocol and other metadata are not model inputs',
      not any(f in column.lower() for column in VALUE_COLS + MASK_COLS + ['t_years'] + STATICS + STATIC_MASK_COLS for f in forbidden))
show(pd.DataFrame(checks), 'Leakage checks', limit=len(checks))

## 7. The GRU-ODE-Bayes Model

The model keeps one hidden state per patient that summarises everything known so far. Between visits a **GRU-ODE** moves the state forward in continuous time, so a two-year gap and a six-month gap are handled by the same dynamics integrated for different durations. At a visit the model first predicts every channel (a mean and a log-variance), then a **GRU-Bayes** cell corrects the state using the prediction errors scaled by the predicted uncertainty, and only the channels that were actually observed take part in the correction. A small network maps the static covariates and their masks to the initial state. We follow De Brouwer et al. (NeurIPS 2019) and keep the model deliberately small for a cohort of about two thousand training patients.

In [ ]:
LOG_2PI = math.log(2 * math.pi)

class GRUODEBayes(nn.Module):
    def __init__(self, n_channels, n_static, hidden_size, ode_step_years, max_substeps, logvar_min, logvar_max):
        super().__init__()
        self.n_channels, self.hidden_size = n_channels, hidden_size
        self.ode_step_years, self.max_substeps = ode_step_years, max_substeps
        self.logvar_min, self.logvar_max = logvar_min, logvar_max
        self.h0_net = nn.Sequential(nn.Linear(2 * n_static, hidden_size), nn.Tanh(), nn.Linear(hidden_size, hidden_size), nn.Tanh())
        self.ode_gate_r, self.ode_gate_z, self.ode_gate_g = (nn.Linear(hidden_size, hidden_size) for _ in range(3))
        bayes_input = 2 * n_channels + hidden_size
        self.bayes_gate_r, self.bayes_gate_z, self.bayes_gate_g = (nn.Linear(bayes_input, hidden_size) for _ in range(3))
        self.head = nn.Linear(hidden_size, 2 * n_channels)

    def initial_state(self, static, static_mask):
        return self.h0_net(torch.cat([static * static_mask, static_mask], dim=1))

    def predict(self, h):
        out = self.head(h)
        return out[:, :self.n_channels], torch.clamp(out[:, self.n_channels:], self.logvar_min, self.logvar_max)

    def ode_derivative(self, h):
        r, z = torch.sigmoid(self.ode_gate_r(h)), torch.sigmoid(self.ode_gate_z(h))
        g = torch.tanh(self.ode_gate_g(r * h))
        return (1 - z) * (g - h)

    def evolve(self, h, delta_t):
        """Move h forward by delta_t years (one value per patient) with the midpoint method."""
        n_steps = int(min(self.max_substeps, max(1, math.ceil(float(delta_t.max()) / self.ode_step_years))))
        dt = (delta_t / n_steps).unsqueeze(1)
        for _ in range(n_steps):
            k1 = self.ode_derivative(h)
            k2 = self.ode_derivative(h + 0.5 * dt * k1)
            h = h + dt * k2
        return h

    def bayes_update(self, h, mean, logvar, values, masks):
        error = (values - mean) * torch.exp(-0.5 * logvar) * masks
        x = torch.cat([error, masks, h], dim=1)
        r, z = torch.sigmoid(self.bayes_gate_r(x)), torch.sigmoid(self.bayes_gate_z(x))
        g = torch.tanh(self.bayes_gate_g(torch.cat([error, masks, r * h], dim=1)))
        h_new = (1 - z) * h + z * g
        observed = (masks.sum(dim=1, keepdim=True) > 0).float()
        return observed * h_new + (1 - observed) * h

    def forward(self, batch):
        """Returns the means and log-variances predicted BEFORE each visit, and the states AFTER each update."""
        times, values, masks = batch['times'], batch['values'], batch['masks']
        h = self.initial_state(batch['static'], batch['static_mask'])
        means, logvars, states = [], [], []
        previous_t = torch.zeros_like(times[:, 0])
        for k in range(times.shape[1]):
            h = self.evolve(h, times[:, k] - previous_t)
            mean, logvar = self.predict(h)
            h = self.bayes_update(h, mean, logvar, values[:, k], masks[:, k])
            means.append(mean); logvars.append(logvar); states.append(h)
            previous_t = times[:, k]
        return torch.stack(means, dim=1), torch.stack(logvars, dim=1), torch.stack(states, dim=1)

    def forecast(self, h, delta_t):
        """Predict every channel delta_t years after state h, without new observations."""
        return self.predict(self.evolve(h, delta_t))

def gaussian_nll(mean, logvar, values, masks):
    """Negative log-likelihood averaged over observed entries only."""
    nll = 0.5 * (logvar + (values - mean) ** 2 * torch.exp(-logvar) + LOG_2PI)
    return (nll * masks).sum() / masks.sum().clamp(min=1.0)

model = GRUODEBayes(len(CHANNELS), len(STATICS), CFG.HIDDEN_SIZE, CFG.ODE_STEP_YEARS, CFG.MAX_SUBSTEPS, CFG.LOGVAR_MIN, CFG.LOGVAR_MAX).to(DEVICE)
PARTS = {'h0_net': 'initial state from static covariates', 'ode': 'GRU-ODE (between visits)',
         'bayes': 'GRU-Bayes (update at a visit)', 'head': 'observation model (mean and log-variance)'}
parameter_table = pd.DataFrame([{'component': label, 'parameters': sum(p.numel() for n, p in model.named_parameters() if n.startswith(key))} for key, label in PARTS.items()])
parameter_table.loc[len(parameter_table)] = ['total', sum(p.numel() for p in model.parameters())]
show(parameter_table, f'Model size: hidden state {CFG.HIDDEN_SIZE}, {len(CHANNELS)} channels, {len(STATICS)} static covariates')

## 8. Training

We train the model to predict each visit before seeing it. The loss is the Gaussian negative log-likelihood of every observed value given the prediction made before the update, so every visit and every measured channel contributes, and the model is rewarded for being accurate and honest about its uncertainty at the same time. We optimise with Adam and gradient clipping, stop when the validation loss has not improved for ten epochs, and keep the best epoch.

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    nll_sum, n_observed = 0.0, 0.0
    abs_error, counts = np.zeros(len(CHANNELS)), np.zeros(len(CHANNELS))
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        with torch.set_grad_enabled(training):
            means, logvars, _ = model(batch)
            loss = gaussian_nll(means, logvars, batch['values'], batch['masks'])
        if training:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            optimizer.step()
        observed = batch['masks'].sum().item()
        nll_sum, n_observed = nll_sum + loss.item() * observed, n_observed + observed
        abs_error += ((means - batch['values']).abs() * batch['masks']).sum(dim=(0, 1)).detach().cpu().numpy()
        counts += batch['masks'].sum(dim=(0, 1)).cpu().numpy()
    return nll_sum / n_observed, abs_error / np.maximum(counts, 1)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
history, best_nll, best_state, best_epoch, waited = [], float('inf'), None, 0, 0
for epoch in range(1, CFG.MAX_EPOCHS + 1):
    train_nll, _ = run_epoch(model, loaders['train'], optimizer)
    val_nll, val_mae = run_epoch(model, loaders['val'])
    history.append({'epoch': epoch, 'train_nll': train_nll, 'val_nll': val_nll,
                    **{f'val_mae_{c}': val_mae[i] * norm['channels'][c]['std'] for i, c in enumerate(CHANNELS)}})
    if val_nll < best_nll:
        best_nll, best_state, best_epoch, waited = val_nll, copy.deepcopy(model.state_dict()), epoch, 0
    else:
        waited += 1
        if waited >= CFG.PATIENCE:
            break
model.load_state_dict(best_state)
torch.save(model.state_dict(), OUT / 'gru_ode_bayes.pt')
history = pd.DataFrame(history)
history.to_csv(OUT / 'training_history.csv', index=False)
show(history[(history.epoch % 10 == 0) | (history.epoch == best_epoch) | (history.epoch == 1)], 'Training history (validation MAE in raw score units)', limit=20)
note(f'Best epoch {best_epoch} with validation NLL {best_nll:.3f}; training stopped after epoch {len(history)}.')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(history.epoch, history.train_nll, color=LIGHT, linewidth=2, label='train')
axes[0].plot(history.epoch, history.val_nll, color=PURPLE, linewidth=2, label='validation')
axes[0].axvline(best_epoch, color=GREY, linestyle=':', linewidth=1)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('NLL per observed value'); axes[0].legend(frameon=False)
style_axis(axes[0], 'Loss (dotted line = best epoch)')
axes[1].plot(history.epoch, history.val_mae_cdrsb, color=PURPLE, linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('CDR-SB points')
style_axis(axes[1], 'Validation MAE for CDR-SB at each visit, predicted before seeing it')
finish(fig, 'training_curves')

## 9. Evaluation

We judge the model on the evaluation examples in three steps. We first compute two baselines that need no training: **no change** repeats the patient's CDR-SB at the index visit, and **personal trend** extends a straight line fitted to the patient's own CDR-SB history. We then produce the model's forecasts, calibrate its prediction intervals on the calibration patients, and finally report accuracy and interval coverage on the test patients, overall and by subgroup.

### 9.1 Baselines and model forecasts

For each example the model reads only the first `n_input_visits` visits, which the leakage checks guarantee end at the index visit. From the state after the index visit the ODE runs forward to the target date without new observations, and the prediction is converted back to CDR-SB points.

In [ ]:
CDR = CHANNELS.index('cdrsb')
CDR_MEAN, CDR_STD = norm['channels']['cdrsb']['mean'], norm['channels']['cdrsb']['std']

def cdr_history(split):
    rows = sequences[split][sequences[split].mask_cdrsb == 1]
    return {rid: (g.t_years.to_numpy(), g.value_cdrsb.to_numpy() * CDR_STD + CDR_MEAN) for rid, g in rows.groupby('RID')}

def add_baselines(frame, split):
    history_by_rid = cdr_history(split)
    no_change, trend = [], []
    for row in frame.itertuples():
        times, values = history_by_rid[row.RID]
        past = times <= row.index_time + 1e-9
        slope, intercept = np.polyfit(times[past], values[past], 1)
        no_change.append(values[past][-1])
        trend.append(float(np.clip(slope * row.target_time + intercept, 0, 18)))
    return frame.assign(pred_no_change=no_change, pred_trend=trend)

def collect_states(split):
    """Hidden state after every visit of every patient in a split."""
    loader = DataLoader(datasets[split], batch_size=CFG.BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    states = {}
    model.eval()
    with torch.no_grad():
        for batch in loader:
            _, _, h = model({key: value.to(DEVICE) for key, value in batch.items()})
            for i, rid in enumerate(batch['rid'].tolist()):
                states[rid] = h[i, :int(batch['lengths'][i])].cpu().numpy()
    return states

def add_forecasts(frame, states, chunk=512):
    means, sigmas = [], []
    with torch.no_grad():
        for start in range(0, len(frame), chunk):
            part = frame.iloc[start:start + chunk]
            h = torch.as_tensor(np.stack([states[rid][k] for rid, k in zip(part.RID, part.index_visit_index)])).to(DEVICE)
            mean, logvar = model.forecast(h, torch.as_tensor(part.delta_years.to_numpy(dtype=np.float32)).to(DEVICE))
            means.append(mean[:, CDR].cpu().numpy() * CDR_STD + CDR_MEAN)
            sigmas.append(np.exp(0.5 * logvar[:, CDR].cpu().numpy()) * CDR_STD)
    return frame.assign(pred_mean=np.concatenate(means), pred_sigma_raw=np.concatenate(sigmas))

states_cache = {}
for split in ['val', 'calib', 'test']:
    states_cache[split] = collect_states(split)
    examples[split] = add_forecasts(add_baselines(examples[split], split), states_cache[split])
show(examples['test'][['RID', 'horizon_months', 'n_input_visits', 'index_cdrsb_raw', 'target_cdrsb_raw', 'pred_no_change', 'pred_trend', 'pred_mean', 'pred_sigma_raw']],
     'Baseline and model forecasts for the first test examples (before calibration)')

### 9.2 Calibrating the uncertainty

A "90% interval" should contain the true value about 90% of the time. On the calibration patients only, we fit one scale factor per horizon that multiplies the predicted standard deviation until the empirical coverage is exact there, interpolate between horizons, and clip intervals to the valid CDR-SB range. The factors are saved with the model; the test patients never influence them.

In [ ]:
Z = gaussian.ppf(0.5 + CFG.COVERAGE / 2)

def coverage(frame, level, sigma_column):
    z = gaussian.ppf(0.5 + level / 2)
    return float(((frame.target_cdrsb_raw - frame.pred_mean).abs() <= z * frame[sigma_column]).mean())

sigma_scale = {}
for h in HORIZONS:
    part = examples['calib'][examples['calib'].horizon_months == h]
    standardized = ((part.target_cdrsb_raw - part.pred_mean).abs() / part.pred_sigma_raw).to_numpy()
    sigma_scale[h] = float(max(np.quantile(standardized, CFG.COVERAGE) / Z, 1e-3))

def sigma_scale_at(delta_years):
    return np.interp(delta_years, np.array(HORIZONS) / 12, [sigma_scale[h] for h in HORIZONS])

for split in ['val', 'calib', 'test']:
    examples[split]['pred_sigma'] = examples[split].pred_sigma_raw * examples[split].horizon_months.map(sigma_scale)
    examples[split]['lower'] = (examples[split].pred_mean - Z * examples[split].pred_sigma).clip(lower=0)
    examples[split]['upper'] = (examples[split].pred_mean + Z * examples[split].pred_sigma).clip(upper=18)

calibration_table = pd.DataFrame([{'horizon_months': h, 'sigma_scale': sigma_scale[h],
    'calib_coverage_before': coverage(examples['calib'][examples['calib'].horizon_months == h], CFG.COVERAGE, 'pred_sigma_raw'),
    'calib_coverage_after': coverage(examples['calib'][examples['calib'].horizon_months == h], CFG.COVERAGE, 'pred_sigma'),
    'test_coverage_before': coverage(examples['test'][examples['test'].horizon_months == h], CFG.COVERAGE, 'pred_sigma_raw'),
    'test_coverage_after': coverage(examples['test'][examples['test'].horizon_months == h], CFG.COVERAGE, 'pred_sigma')} for h in HORIZONS])
show(calibration_table, f'Calibration of the {CFG.COVERAGE:.0%} interval (scale fitted on calibration patients, checked on test patients)')

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(11, 3.4), sharey=True)
for ax, h in zip(axes, HORIZONS):
    part = examples['test'][examples['test'].horizon_months == h]
    ax.plot([0.4, 1], [0.4, 1], color=GREY, linestyle=':', linewidth=1)
    ax.plot(CFG.RELIABILITY_LEVELS, [coverage(part, l, 'pred_sigma_raw') for l in CFG.RELIABILITY_LEVELS], marker='o', color=LIGHT, linewidth=2, label='before calibration')
    ax.plot(CFG.RELIABILITY_LEVELS, [coverage(part, l, 'pred_sigma') for l in CFG.RELIABILITY_LEVELS], marker='o', color=PURPLE, linewidth=2, label='after calibration')
    ax.set_xlabel('Nominal interval level'); style_axis(ax, f'{h} months (test patients)')
axes[0].set_ylabel('Share of true values inside the interval'); axes[0].legend(frameon=False, fontsize=8)
finish(fig, 'reliability')

### 9.3 Results on the test patients

We now report, once, how good the forecasts are: mean absolute error and RMSE in CDR-SB points for the model and both baselines, and the coverage and width of the calibrated 90% interval. Subgroups use metadata only; the model never saw them.

In [ ]:
def score(frame):
    error = frame.target_cdrsb_raw - frame.pred_mean
    return {'examples': len(frame), 'patients': frame.RID.nunique(),
            'mae_model': error.abs().mean(), 'rmse_model': np.sqrt((error ** 2).mean()),
            'mae_no_change': (frame.target_cdrsb_raw - frame.pred_no_change).abs().mean(),
            'mae_trend': (frame.target_cdrsb_raw - frame.pred_trend).abs().mean(),
            f'coverage_{CFG.COVERAGE:.0%}': coverage(frame, CFG.COVERAGE, 'pred_sigma'),
            'interval_width': float((frame.upper - frame.lower).mean())}

def results_by(frame, column):
    return pd.DataFrame([{column: key, 'horizon_months': h, **score(g)} for h in HORIZONS for key, g in frame[frame.horizon_months == h].groupby(column, observed=True)])

test = examples['test']
test_results = pd.DataFrame([{'horizon_months': h, **score(test[test.horizon_months == h])} for h in HORIZONS])
val_results = pd.DataFrame([{'horizon_months': h, **score(examples['val'][examples['val'].horizon_months == h])} for h in HORIZONS])
test_results.to_csv(OUT / 'metrics_test.csv', index=False); val_results.to_csv(OUT / 'metrics_val.csv', index=False)
show(test_results, 'Test results by horizon (CDR-SB points)')
show(val_results, 'Validation results by horizon, for reference')

fig, ax = plt.subplots(figsize=(7, 3.4))
width, x = 0.26, np.arange(len(HORIZONS))
for offset, column, color, label in [(-width, 'mae_model', PURPLE, 'GRU-ODE-Bayes'), (0, 'mae_no_change', LIGHT, 'no change'), (width, 'mae_trend', GREY, 'personal trend')]:
    bars = ax.bar(x + offset, test_results[column], width * 0.92, color=color, label=label)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color=INK)
ax.set_xticks(x); ax.set_xticklabels([f'{h} months' for h in HORIZONS]); ax.set_ylabel('Mean absolute error (CDR-SB points)')
ax.legend(frameon=False, fontsize=9); style_axis(ax, 'Forecast error on test patients: model vs baselines')
finish(fig, 'mae_by_horizon')

In [ ]:
test['history_bin'] = pd.cut(test.n_input_visits, bins=[0, 3, 6, np.inf], labels=['2-3 visits', '4-6 visits', '7+ visits'])
by_protocol, by_history = results_by(test, 'META_baseline_protocol'), results_by(test, 'history_bin')
pd.concat([by_protocol.rename(columns={'META_baseline_protocol': 'group'}).assign(grouping='baseline_protocol'),
           by_history.rename(columns={'history_bin': 'group'}).assign(grouping='history')]).to_csv(OUT / 'metrics_test_subgroups.csv', index=False)
columns = ['horizon_months', 'examples', 'patients', 'mae_model', 'mae_no_change', 'mae_trend', f'coverage_{CFG.COVERAGE:.0%}']
show(by_protocol[['META_baseline_protocol'] + columns], 'Test results by enrolment protocol', limit=20)
show(by_history[['history_bin'] + columns], 'Test results by amount of history before the index visit', limit=12)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
part = test[test.horizon_months == 12]
top = float(max(part.target_cdrsb_raw.max(), part.pred_mean.max())) + 1
axes[0].plot([0, top], [0, top], color=GREY, linestyle=':', linewidth=1)
axes[0].scatter(part.target_cdrsb_raw, part.pred_mean, s=14, color=PURPLE, alpha=0.45, edgecolor='none')
axes[0].set_xlim(-0.5, top); axes[0].set_ylim(-0.5, top)
axes[0].set_xlabel('Actual CDR-SB 12 months later'); axes[0].set_ylabel('Forecast CDR-SB')
style_axis(axes[0], 'Forecast vs actual at 12 months (test patients)')
for h, color in zip(HORIZONS, [LIGHT, MID, PURPLE]):
    part = by_history[by_history.horizon_months == h]
    axes[1].plot(part.history_bin.astype(str), part.mae_model, marker='o', color=color, linewidth=2, label=f'model, {h} m')
    axes[1].plot(part.history_bin.astype(str), part.mae_no_change, marker='o', color=color, linewidth=1.2, linestyle='--', alpha=0.7)
axes[1].set_ylabel('Mean absolute error (points)'); axes[1].legend(frameon=False, fontsize=8)
style_axis(axes[1], 'Error vs amount of history (solid = model, dashed = no change)')
finish(fig, 'test_scatter_and_history')

### 9.4 Example forecasts

Finally we look at individual test patients: the model reads the visits up to an index visit, and we draw the forecast mean and the calibrated 90% band forward in time next to the visits that actually followed. Filled markers are visits the model saw; hollow markers are the future.

In [ ]:
def forecast_curve(split, rid, index_visit_index, horizon_years, steps=30):
    h = torch.as_tensor(states_cache[split][rid][index_visit_index][None]).to(DEVICE)
    grid = np.linspace(0.05, horizon_years, steps).astype(np.float32)
    means, sigmas = [], []
    with torch.no_grad():
        for delta in grid:
            mean, logvar = model.forecast(h, torch.as_tensor(np.array([delta], dtype=np.float32)).to(DEVICE))
            means.append(float(mean[0, CDR]) * CDR_STD + CDR_MEAN)
            sigmas.append(float(torch.exp(0.5 * logvar[0, CDR])) * CDR_STD * sigma_scale_at(delta))
    return grid, np.array(means), np.array(sigmas)

test_history = cdr_history('test')
candidates = test[(test.horizon_months == 24) & (test.n_input_visits >= 3)].sort_values(['RID', 'index_time']).drop_duplicates('RID')
chosen = candidates.iloc[np.linspace(0, len(candidates) - 1, 4).astype(int)]
fig, axes = plt.subplots(1, len(chosen), figsize=(11, 3.4), sharey=True)
for ax, row in zip(np.atleast_1d(axes), chosen.itertuples()):
    times, values = test_history[row.RID]
    seen = times <= row.index_time + 1e-9
    grid, mean, sigma = forecast_curve('test', row.RID, row.index_visit_index, max(times.max() - row.index_time, 2.0) + 0.3)
    ax.fill_between(row.index_time + grid, np.clip(mean - Z * sigma, 0, 18), np.clip(mean + Z * sigma, 0, 18), color=BAND, label='90% interval')
    ax.plot(row.index_time + grid, mean, color=PURPLE, linewidth=2, label='forecast')
    ax.plot(times[seen], values[seen], marker='o', color=INK, linewidth=1, markersize=5, label='seen visits')
    ax.plot(times[~seen], values[~seen], marker='o', markerfacecolor='white', color=INK, linewidth=0, markersize=5, label='later visits')
    ax.set_xlabel('Years since first visit'); style_axis(ax, f'RID {row.RID}, forecast from visit {row.index_visit_index + 1}')
np.atleast_1d(axes)[0].set_ylabel('CDR-SB'); np.atleast_1d(axes)[0].legend(frameon=False, fontsize=7)
finish(fig, 'example_forecasts')

## 10. The Digital Twin in Use

To make the model usable beyond this notebook, we save the weights together with the architecture, the normalization and the calibration parameters, and wrap them in a small `ReplicaForecaster` class that accepts raw clinical scores and returns calibrated forecasts. We then show the defining behaviour of a digital twin: for one test patient, the forecast is recomputed after the second, third and fourth visit, and the band narrows as evidence accumulates.

In [ ]:
model_config = {'architecture': {'hidden_size': CFG.HIDDEN_SIZE, 'ode_step_years': CFG.ODE_STEP_YEARS, 'max_substeps': CFG.MAX_SUBSTEPS,
                                 'logvar_min': CFG.LOGVAR_MIN, 'logvar_max': CFG.LOGVAR_MAX},
                'channel_order': CHANNELS, 'static_order': STATICS, 'normalization': norm,
                'calibration': {'coverage': CFG.COVERAGE, 'z': float(Z), 'sigma_scale_by_horizon_months': {str(h): sigma_scale[h] for h in HORIZONS},
                                'valid_range': {c: list(RANGES[c]) for c in CHANNELS}},
                'training': {'config': vars(CFG), 'best_epoch': int(best_epoch), 'best_val_nll': float(best_nll), 'epochs_run': int(len(history))},
                'library_versions': {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__, 'pandas': pd.__version__}}
(OUT / 'model_config.json').write_text(json.dumps(model_config, indent=2, default=str), encoding='utf-8')
for split in ['val', 'calib', 'test']:
    examples[split].to_csv(OUT / f'predictions_{split}.csv', index=False, date_format='%Y-%m-%d', float_format='%.6f')

class ReplicaForecaster:
    """Load a saved model folder and forecast raw-scale CDR-SB for one patient from raw clinical values."""
    def __init__(self, folder):
        folder = Path(folder)
        self.spec = json.loads((folder / 'model_config.json').read_text())
        self.channels, self.statics = self.spec['channel_order'], self.spec['static_order']
        normalization = self.spec['normalization']
        self.channel_mean = np.array([normalization['channels'][c]['mean'] for c in self.channels], dtype=np.float32)
        self.channel_std = np.array([normalization['channels'][c]['std'] for c in self.channels], dtype=np.float32)
        self.static_mean = np.array([normalization['static'].get(s, {'mean': 0.0})['mean'] for s in self.statics], dtype=np.float32)
        self.static_std = np.array([normalization['static'].get(s, {'std': 1.0})['std'] for s in self.statics], dtype=np.float32)
        self.sigma_scale, self.z = self.spec['calibration']['sigma_scale_by_horizon_months'], self.spec['calibration']['z']
        arch = self.spec['architecture']
        self.model = GRUODEBayes(len(self.channels), len(self.statics), arch['hidden_size'], arch['ode_step_years'], arch['max_substeps'], arch['logvar_min'], arch['logvar_max'])
        self.model.load_state_dict(torch.load(folder / 'gru_ode_bayes.pt', map_location='cpu'))
        self.model.eval()

    def _calibration(self, delta_years):
        horizons = np.array(sorted(int(h) for h in self.sigma_scale))
        return float(np.interp(delta_years, horizons / 12, [self.sigma_scale[str(h)] for h in horizons]))

    def forecast(self, times, raw_values, masks, static_raw, static_mask, future_times, channel='cdrsb'):
        """Rows with mean, sigma and the calibrated interval for `channel` at each future time (years since first visit)."""
        times, masks = np.asarray(times, dtype=np.float32), np.asarray(masks, dtype=np.float32)
        values = np.where(masks > 0, (np.asarray(raw_values, dtype=np.float32) - self.channel_mean) / self.channel_std, 0.0).astype(np.float32)
        static_mask = np.asarray(static_mask, dtype=np.float32)
        static = np.where(static_mask > 0, (np.asarray(static_raw, dtype=np.float32) - self.static_mean) / self.static_std, 0.0).astype(np.float32)
        batch = {'times': torch.as_tensor(times[None]), 'values': torch.as_tensor(values[None]), 'masks': torch.as_tensor(masks[None]),
                 'static': torch.as_tensor(static[None]), 'static_mask': torch.as_tensor(static_mask[None])}
        c = self.channels.index(channel)
        lower_bound, upper_bound = self.spec['calibration']['valid_range'].get(channel, [-np.inf, np.inf])
        rows = []
        with torch.no_grad():
            _, _, states = self.model(batch)
            h = states[:, -1]
            for future in np.asarray(future_times, dtype=np.float32):
                delta = float(future - times[-1])
                mean, logvar = self.model.forecast(h, torch.as_tensor(np.array([delta], dtype=np.float32)))
                mu = float(mean[0, c]) * self.channel_std[c] + self.channel_mean[c]
                sigma = float(torch.exp(0.5 * logvar[0, c])) * self.channel_std[c] * self._calibration(delta)
                rows.append({'future_time': float(future), 'delta_years': delta, 'mean': mu, 'sigma': sigma,
                             'lower_90': max(mu - self.z * sigma, lower_bound), 'upper_90': min(mu + self.z * sigma, upper_bound)})
        return rows

twin = ReplicaForecaster(OUT)

In [ ]:
rid = int(chosen.RID.iloc[0])
visits = sequences['test'][sequences['test'].RID == rid].sort_values('t_years')
patient_static = statics['test'][statics['test'].RID == rid].iloc[0]
channel_mean, channel_std = np.array([norm['channels'][c]['mean'] for c in CHANNELS]), np.array([norm['channels'][c]['std'] for c in CHANNELS])
static_mean = np.array([norm['static'].get(s, {'mean': 0.0})['mean'] for s in STATICS]); static_std = np.array([norm['static'].get(s, {'std': 1.0})['std'] for s in STATICS])
raw_values = visits[VALUE_COLS].to_numpy() * channel_std + channel_mean
raw_static = patient_static[STATICS].to_numpy(dtype=float) * static_std + static_mean
times, cdr_mask = visits.t_years.to_numpy(), visits.mask_cdrsb.to_numpy() == 1
future = np.linspace(times[1] + 0.05, times.max() + 1.0, 40)

fig, ax = plt.subplots(figsize=(8, 3.6))
for n_seen, color in zip([n for n in [2, 3, 4] if n < len(times)], [LIGHT, MID, PURPLE]):
    rows = pd.DataFrame(twin.forecast(times[:n_seen], raw_values[:n_seen], visits[MASK_COLS].to_numpy()[:n_seen],
                                      raw_static, patient_static[STATIC_MASK_COLS].to_numpy(dtype=float), future[future > times[n_seen - 1]]))
    ax.fill_between(rows.future_time, rows.lower_90, rows.upper_90, color=color, alpha=0.18)
    ax.plot(rows.future_time, rows['mean'], color=color, linewidth=2, label=f'forecast after visit {n_seen}')
ax.plot(times[cdr_mask], raw_values[cdr_mask, CDR], marker='o', color=INK, linewidth=1, markersize=5, label='observed CDR-SB')
ax.set_xlabel('Years since first visit'); ax.set_ylabel('CDR-SB'); ax.legend(frameon=False, fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5))
style_axis(ax, f'RID {rid}: the forecast updates as visits arrive')
finish(fig, 'updating_forecast')

In [ ]:
# A new (hypothetical) patient: three visits at 0, 6 and 12 months, raw scores in channel order CDR-SB, MMSE, ADAS-13, FAQ.
new_times = [0.0, 0.5, 1.0]
new_values = [[1.5, 27, 14, 3], [2.0, 26, 16, 0], [2.5, 25, 18, 6]]
new_masks = [[1, 1, 1, 1], [1, 1, 1, 0], [1, 1, 1, 1]]          # FAQ was not measured at the second visit
new_static, new_static_mask = [73, 1, 16, 0], [1, 1, 1, 1]      # age at first visit, female, years of education, APOE4 alleles
forecast_table = pd.DataFrame(twin.forecast(new_times, new_values, new_masks, new_static, new_static_mask, future_times=[1.5, 2.0, 3.0]))
show(forecast_table, 'Calibrated CDR-SB forecast for a new patient at 6, 12 and 24 months after the last visit')

with zipfile.ZipFile(f'{OUT}.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob('*'):
        if path.is_file() and 'extracted' not in path.relative_to(OUT).parts:
            archive.write(path, path.relative_to(OUT))
note(f'All outputs (weights, model_config.json, predictions, metrics, figures and the cleaned ledger) are saved in {OUT} and zipped as {OUT}.zip.')

## 11. Conclusion

We built REPLICA end to end: 22 raw ADNI tables were cleaned without imputation and with full provenance, turned into leakage-free patient sequences, and used to train a compact GRU-ODE-Bayes model that forecasts CDR-SB in continuous time. The summary below compares the model with the stronger of the two baselines at every horizon, alongside the coverage and width of its calibrated 90% interval.

In [ ]:
summary = test_results.assign(best_baseline_mae=test_results[['mae_no_change', 'mae_trend']].min(axis=1))
summary['improvement_over_best_baseline'] = 1 - summary.mae_model / summary.best_baseline_mae
show(summary[['horizon_months', 'examples', 'patients', 'mae_model', 'best_baseline_mae', 'improvement_over_best_baseline', f'coverage_{CFG.COVERAGE:.0%}', 'interval_width']],
     'Test summary: GRU-ODE-Bayes against the best baseline at each horizon')

**Limitations we carry into the report**

- The data are ADNI only and retrospective; the model has not been validated on a Saudi population, and the cognitive tests are sensitive to language, culture and education.
- Evaluation examples require a future CDR-SB observation, so patients lost to follow-up are under-represented.
- The diagnosis is assumed known at the visit; it selects evaluation examples and stratifies the split but never enters the model.
- Uncertainty is Gaussian, calibrated per horizon on one held-out group and interpolated between horizons; intervals are clipped to 0–18.
- MRI, CSF and plasma biomarkers are cleaned and available in the ledger but not used in this version.

**Next steps.** Add the imaging and biomarker channels, tune the model size and training settings on the validation patients, replace the Gaussian calibration with conformal prediction, and validate externally, ultimately on local clinical data.